In [1]:
# Install library
!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pip install -q unsloth langgraph langchain sentence-transformers faiss-cpu pydantic

# Install ngrok (untuk tunneling)
!wget -q -O ngrok.zip https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -o ngrok.zip
!chmod +x ngrok

#import token ngrok
!ngrok config add-authtoken 3DdBpUEORvSaoQWXzTMehc7JGhY_7kAvnMWYZ2vDEeeHTM6FN

Archive:  ngrok.zip
  inflating: ngrok                   
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [2]:
import json, time, torch, faiss, numpy as np
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage
from typing import TypedDict, Annotated, List, Dict, Literal
from pydantic import BaseModel, Field

# ========== 1. Load Model ==========
model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# ========== 2. Tools ==========
WMS_DATA = {
    "Produk_X": {"stock_available": 3871, "machine_capacity_per_day": 1311},
    "Produk_Y": {"stock_available": 17598, "machine_capacity_per_day": 1291},
    "Produk_Z": {"stock_available": 13668, "machine_capacity_per_day": 1763},
    "Alpha_Part": {"stock_available": 17891, "machine_capacity_per_day": 1044},
    "Beta_Core": {"stock_available": 4808, "machine_capacity_per_day": 596},
}

def cek_stok(product_name: str, quantity: int) -> Dict:
    if product_name not in WMS_DATA:
        return {"status": "error", "message": f"Produk {product_name} tidak ditemukan"}
    data = WMS_DATA[product_name]
    available = data["stock_available"]
    can_fulfill = available >= quantity
    return {
        "status": "success",
        "product": product_name,
        "stock_available": available,
        "requested": quantity,
        "can_fulfill": can_fulfill,
        "machine_capacity_per_day": data["machine_capacity_per_day"],
        "message": f"Stok {product_name}: {available} unit" if can_fulfill else f"Stok tidak cukup. Tersedia {available}"
    }

def cek_kapasitas(product_name: str, days: int) -> Dict:
    if product_name not in WMS_DATA:
        return {"status": "error", "message": f"Produk {product_name} tidak ditemukan"}
    cap_per_day = WMS_DATA[product_name]["machine_capacity_per_day"]
    max_prod = cap_per_day * days
    return {
        "status": "success",
        "product": product_name,
        "capacity_per_day": cap_per_day,
        "days_available": days,
        "max_production": max_prod,
        "message": f"Kapasitas: {cap_per_day}/hari, max {max_prod} dalam {days} hari"
    }

def draft_po(product_name: str, quantity: int, supplier: str = "Supplier Utama") -> Dict:
    from datetime import datetime
    po_number = f"PO-{product_name}-{quantity}-{datetime.now().strftime('%Y%m%d')}"
    return {
        "status": "success",
        "po_draft": {
            "po_number": po_number,
            "product": product_name,
            "quantity": quantity,
            "supplier": supplier,
            "estimated_lead_time": "3-5 hari kerja",
            "status": "DRAFT"
        },
        "message": f"Draft PO untuk {quantity} unit {product_name} telah dibuat"
    }

# ========== 3. RAG ==========
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
SOP_DATA = [
    {"id": "sop_001", "category": "negotiation", "content": "Untuk pesanan yang ditolak karena kapasitas, Sales Agent wajib menawarkan opsi pengiriman bertahap minimal 2 tahap sebelum menyatakan pesanan tidak dapat dipenuhi."},
    {"id": "sop_002", "category": "negotiation", "content": "Jika pesanan ditolak karena stok, tawarkan produk alternatif atau kurangi quantity menjadi 50% dari permintaan awal."}
]
sop_texts = [doc["content"] for doc in SOP_DATA]
sop_embeddings = embedding_model.encode(sop_texts, convert_to_numpy=True)
dimension = sop_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(sop_embeddings)
index.add(sop_embeddings)

def retrieve_sop(query: str, top_k=1) -> str:
    q_emb = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, top_k)
    docs = []
    for i in indices[0]:
        if i < len(sop_texts):
            docs.append(sop_texts[i])
    return "\n".join(docs) if docs else ""

# ========== 4. LLM Helpers & Prompts ==========
def call_llm(system_prompt: str, user_message: str, max_tokens=128) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.2, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    return response

SALES_CAPACITY_PROMPT = """Anda adalah Sales & Capacity Agent terintegrasi.
Tugas: Evaluasi pesanan berikut. Cek stok dan kapasitas, lalu putuskan status:
- approved: jika stok >= quantity dan kapasitas >= quantity
- conditional: jika stok cukup tapi kapasitas kurang (tawarkan pengiriman bertahap)
- rejected: jika stok tidak cukup
Berikan alasan dengan angka (misal "Stok 3871, kapasitas 6555").
Output JSON: {"status": "...", "reason": "...", "recommendation": "..."}"""

NEGOTIATION_PROMPT = """Anda adalah Sales Negotiator.
Kapasitas menolak atau hanya menyetujui sebagian. Gunakan SOP berikut jika relevan:
{sop_context}
Buat tawaran alternatif (pengiriman bertahap) atau saran kurangi quantity.
Output JSON: {"status": "conditional"|"rejected", "message": "...", "proposed_terms": {...}}"""

# ========== 5. LangGraph Agent ==========
class CapacityDecision(BaseModel):
    status: Literal["approved", "conditional", "rejected"]
    reason: str
    recommendation: str

def call_llm_structured(system_prompt: str, user_message: str, max_tokens=128) -> CapacityDecision:
    try:
        response = call_llm(system_prompt, user_message, max_tokens)
        return CapacityDecision.model_validate_json(response)
    except Exception as e:
        print(f"[LLM Structured] Error: {e}")
        return None

class AgentState(TypedDict):
    messages: Annotated[List[Dict], add_messages]
    order_data: Dict
    capacity_result: Dict
    capacity_decision: Dict
    sales_decision: Dict
    procurement_action: Dict
    final_response: Dict
    pending_approval: Dict

# ========== PERBAIKAN UTAMA DI SINI ==========
def sales_capacity_node(state: AgentState) -> dict:
    print("[Sales-Capacity] Processing order...")
    order = state.get("order_data", {})
    product = order.get("product", "")
    quantity = order.get("quantity", 0)
    deadline = order.get("deadline", 7)

    stock_result = cek_stok(product, quantity)
    cap_result = cek_kapasitas(product, deadline)

    capacity_res = {
        "product": product,
        "quantity": quantity,
        "deadline": deadline,
        "stock_available": stock_result.get("stock_available", 0),
        "capacity_per_day": cap_result.get("capacity_per_day", 0),
        "max_production": cap_result.get("max_production", 0),
        "can_fulfill_stock": stock_result.get("can_fulfill", False),
        "can_fulfill_capacity": cap_result.get("max_production", 0) >= quantity
    }

    user_msg = f"Produk {product}, qty {quantity}, deadline {deadline} hari. Stok={capacity_res['stock_available']}, Kapasitas={capacity_res['max_production']}."

    # 🔧 Bungkus panggilan LLM dengan try-except
    try:
        decision_obj = call_llm_structured(SALES_CAPACITY_PROMPT, user_msg, max_tokens=128)
        if decision_obj:
            decision = decision_obj.model_dump()
        else:
            decision = None
    except Exception as e:
        print(f"[Sales-Capacity] LLM error: {e}")
        decision = None

    # Fallback jika LLM gagal
    if decision is None:
        if capacity_res["can_fulfill_stock"] and capacity_res["can_fulfill_capacity"]:
            decision = {"status": "approved", "reason": "Stok & kapasitas cukup", "recommendation": "Lanjutkan"}
        elif capacity_res["can_fulfill_stock"]:
            decision = {"status": "conditional", "reason": "Kapasitas kurang", "recommendation": "Tawarkan bertahap"}
        else:
            decision = {"status": "rejected", "reason": "Stok tidak cukup", "recommendation": "Tolak"}

    return {
        "capacity_result": capacity_res,
        "capacity_decision": decision,
        "messages": [AIMessage(content=json.dumps(decision), name="SalesCapacityAgent")]
    }

def negotiation_node(state: AgentState) -> dict:
    cap_dec = state.get("capacity_decision", {})
    order = state.get("order_data", {})
    product = order.get("product", "")
    quantity = order.get("quantity", 0)

    if isinstance(cap_dec, dict) and cap_dec.get("status") == "approved":
        return {
            "sales_decision": {"status": "approved", "message": "Disetujui"},
            "final_response": {"status": "approved", "message": f"Order {product} x {quantity} APPROVED"}
        }

    query = f"Pesanan ditolak karena {cap_dec.get('reason', '') if isinstance(cap_dec, dict) else ''}"
    sop_context = retrieve_sop(query, top_k=1)

    print("[Negotiation] Auto-negotiation with RAG...")
    prompt = f"Status capacity: {json.dumps(cap_dec)}. Pesanan: {product} {quantity} unit."
    filled_prompt = NEGOTIATION_PROMPT.format(sop_context=sop_context)

    try:
        response = call_llm(filled_prompt, prompt, max_tokens=128)
        import re
        json_match = re.search(r'\{.*\}', response.replace('\n', ''))
        if json_match:
            result = json.loads(json_match.group())
        else:
            result = json.loads(response)

        if not isinstance(result, dict):
            raise ValueError("Bukan dictionary")
        if "status" not in result:
            result["status"] = "rejected"
    except Exception as e:
        print(f"[Negotiation] Fallback karena: {e}")
        if isinstance(cap_dec, dict) and cap_dec.get("status") == "conditional":
            first = quantity // 2
            result = {"status": "conditional", "message": f"Tawaran bertahap: {first} sekarang, sisanya nanti."}
        else:
            result = {"status": "rejected", "message": "Tidak bisa dipenuhi."}

    return {
        "sales_decision": result,
        "final_response": {
            "status": result.get("status", "rejected"),
            "message": result.get("message", "Ditolak otomatis")
        },
        "messages": [AIMessage(content=str(result), name="Negotiator")]
    }

def procurement_node(state: AgentState) -> dict:
    print("[Procurement] Monitoring stock...")
    order = state.get("order_data", {})
    product = order.get("product", "")
    quantity = order.get("quantity", 0)

    stock_res = cek_stok(product, 0)
    stock_avail = stock_res.get("stock_available", 0)

    final_resp = state.get("final_response", {})
    final_status = final_resp.get("status") if isinstance(final_resp, dict) else None

    if final_status == "approved":
        remaining = stock_avail - quantity
    else:
        remaining = stock_avail

    if remaining < 1000 or remaining < stock_avail * 0.2:
        pending_app = {"product": product, "quantity": quantity, "remaining": remaining, "stock_avail": stock_avail}
        print(f"[HITL] Approval needed for PO: {quantity} units of {product}")
        result = {"status": "pending_approval", "message": f"PO {quantity} unit menunggu approval."}
    else:
        pending_app = {}
        result = {"status": "no_action_needed", "message": "Stok aman"}

    return {
        "pending_approval": pending_app,
        "procurement_action": result,
        "messages": [AIMessage(content=json.dumps(result), name="ProcurementAgent")]
    }

def route_after_negotiation(state: AgentState) -> str:
    sales_dec = state.get("sales_decision", {})
    if isinstance(sales_dec, dict) and sales_dec.get("status") in ["approved", "conditional"]:
        return "procurement"
    return END

workflow = StateGraph(AgentState)
workflow.add_node("sales_capacity", sales_capacity_node)
workflow.add_node("negotiation", negotiation_node)
workflow.add_node("procurement", procurement_node)
workflow.set_entry_point("sales_capacity")
workflow.add_edge("sales_capacity", "negotiation")
workflow.add_conditional_edges("negotiation", route_after_negotiation, {"procurement": "procurement", END: END})
workflow.add_edge("procurement", END)
AGENT_GRAPH = workflow.compile()

def run_agent(order_data: Dict) -> Dict:
    start = time.time()
    product = order_data.get("product", "")
    quantity = order_data.get("quantity", 0)
    deadline = order_data.get("deadline", 7)

    # ==========================================================
    # 1. PRE-CALCULATION (FAIL-SAFE)
    # ==========================================================
    stock_res = cek_stok(product, quantity)
    cap_res = cek_kapasitas(product, deadline)

    fallback_capacity_result = {
        "product": product,
        "quantity": quantity,
        "deadline": deadline,
        "stock_available": stock_res.get("stock_available", 0),
        "capacity_per_day": cap_res.get("capacity_per_day", 0),
        "max_production": cap_res.get("max_production", 0),
        "can_fulfill_stock": stock_res.get("can_fulfill", False),
        "can_fulfill_capacity": cap_res.get("max_production", 0) >= quantity
    }

    initial_state = {
        "messages": [],
        "order_data": order_data,
        "capacity_result": fallback_capacity_result,
        "capacity_decision": {},
        "sales_decision": {},
        "procurement_action": {},
        "final_response": {},
        "pending_approval": {}
    }

    config = {"configurable": {"thread_id": f"thread_{int(time.time())}"}}

    try:
        # ==========================================================
        # 2. JALANKAN LANGGRAPH AI
        # ==========================================================
        result = AGENT_GRAPH.invoke(initial_state, config=config)

    except Exception as e:
        import traceback
        traceback.print_exc()
        error_msg = str(e)

        # JIKA AI CRASH: Kembalikan fallback DENGAN FORMAT MULTI-AGENT
        return {
            "order": order_data,
            "status": "rejected",
            "agents_interaction": {
                "sales_agent": {
                    "role": "Order Ingestion & Negotiation",
                    "initial_action": f"Menerima pesanan {quantity} unit {product}. Meminta validasi dari Capacity Agent.",
                    "negotiation_action": "Sistem otomatis membatalkan negosiasi karena AI tidak merespon."
                },
                "capacity_agent": {
                    "role": "Operational Gatekeeper",
                    "decision": "Error / Bypass",
                    "reasoning": f"AI Internal Error: {error_msg}. Kapasitas gagal dianalisis secara semantik.",
                    "remaining_stock": fallback_capacity_result["stock_available"]
                },
                "procurement_agent": {
                    "role": "Supply Chain Executor",
                    "status": "Skipped",
                    "action": "Sistem terhenti karena kendala AI. Tidak ada PO diterbitkan."
                }
            },
            "elapsed_seconds": round(time.time() - start, 2)
        }

    elapsed = time.time() - start

    # ==========================================================
    # 3. PROTEKSI TIPE DATA
    # ==========================================================
    def safe_dict(val):
        return val if isinstance(val, dict) else {}

    sales_dec = safe_dict(result.get("sales_decision"))
    proc_action = safe_dict(result.get("procurement_action"))
    cap_dec = safe_dict(result.get("capacity_decision"))

    cap_result = safe_dict(result.get("capacity_result"))
    if not cap_result:
        cap_result = fallback_capacity_result

    final_status = sales_dec.get("status", "unknown")

    # ==========================================================
    # 4. EKSEKUSI PENGURANGAN STOK GUDANG
    # ==========================================================
    if product in WMS_DATA:
        try:
            qty_int = int(quantity)
            if final_status == "approved":
                WMS_DATA[product]["stock_available"] -= qty_int
            elif final_status == "conditional":
                WMS_DATA[product]["stock_available"] = 0
        except ValueError:
            pass
        cap_result["stock_available"] = WMS_DATA[product]["stock_available"]

    # ==========================================================
    # 5. MAPPING KE STRUKTUR FRONTEND (AGENTS INTERACTION)
    # ==========================================================
    return {
        "order": order_data,
        "status": final_status,
        "agents_interaction": {
            "sales_agent": {
                "role": "Order Ingestion & Negotiation",
                "initial_action": f"Menerima pesanan {quantity} unit {product} (Deadline: {deadline} hari). Mengirim request persetujuan ke Capacity Agent.",
                "negotiation_action": sales_dec.get("message", "Pesanan langsung diteruskan, tidak ada negosiasi tambahan.")
            },
            "capacity_agent": {
                "role": "Operational Gatekeeper",
                "decision": cap_dec.get("status", "Unknown").capitalize(),
                "reasoning": cap_dec.get("reason", "Disetujui secara otomatis berdasarkan parameter dasar."),
                "remaining_stock": cap_result.get("stock_available", 0)
            },
            "procurement_agent": {
                "role": "Supply Chain Executor",
                "status": proc_action.get("status", "Standby").replace("_", " ").title(),
                "action": proc_action.get("message", "Stok pasca-transaksi masih dalam batas aman.")
            }
        },
        "elapsed_seconds": round(elapsed, 2)
    }

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import time
import os
import subprocess

# ========== Terapkan nest_asyncio ==========
nest_asyncio.apply()

# ========== Buat FastAPI app ==========
app = FastAPI(title="Multi-Agent Sales & Procurement System")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class OrderRequest(BaseModel):
    product: str
    quantity: int
    deadline: int = 7
    client: str = "Default Client"
    urgency: str = "medium"

@app.post("/process_order")
async def process_order(order: OrderRequest):
    try:
        order_data = order.dict()
        result = run_agent(order_data)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "ok", "timestamp": time.time()}

# ========== Hentikan proses yang menggunakan port 8000 ==========
# Cara lebih andal di Colab: gunakan fuser (jika ada) atau kill dengan lsof -t
try:
    # coba fuser
    os.system("fuser -k 8000/tcp 2>/dev/null")
except:
    pass
# alternatif
os.system("lsof -ti:8000 | xargs kill -9 2>/dev/null")

# ========== Jalankan server di thread dengan penanganan error ==========
def run_server():
    try:
        uvicorn.run(
            app,
            host="0.0.0.0",
            port=8000,
            log_level="debug",   # agar error terlihat
            loop="asyncio"
        )
    except Exception as e:
        print(f"❌ Uvicorn gagal: {e}")
        # Jika gagal, kita bisa coba port lain atau berhenti
        raise  # biarkan error muncul di thread utama

# Gunakan thread non-daemon agar error tidak disembunyikan
server_thread = threading.Thread(target=run_server, daemon=False)
server_thread.start()

# Tunggu sebentar agar server siap
time.sleep(3)
print("✅ Server started on port 8000")

# ========== Expose dengan ngrok ==========
try:
    # Hentikan tunnel sebelumnya jika ada
    ngrok.kill()
    public_url = ngrok.connect(8000)
    print(f"🔗 Public URL: {public_url}")
except Exception as e:
    print(f"❌ Ngrok error: {e}")
    print("Periksa authtoken Anda.")

INFO:     Started server process [17571]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Server started on port 8000
🔗 Public URL: NgrokTunnel: "https://brethren-backspace-browse.ngrok-free.dev" -> "http://localhost:8000"


In [4]:
import requests

# Ganti dengan URL public dari output cell sebelumnya
url = "https://brethren-backspace-browse.ngrok-free.dev/health"
try:
    resp = requests.get(url)
    print("Health check:", resp.json())
except Exception as e:
    print("Error:", e)

INFO:     34.21.171.27:0 - "GET /health HTTP/1.1" 200 OK
Health check: {'status': 'ok', 'timestamp': 1783516489.9469566}
